# Sample Repeatability
Implemented Randomization


#### See Plots in data/output

The thick horizontal line inside each violin (from the sns.boxplot overlay) is the median (50th percentile) of the pairwise‐distance distribution for that ROI.

The red “X” marker (added with sns.pointplot) is the mean of the same distribution.

# Cortical Correlation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize

# subj_list = ["subj_01"]  # adjust as needed
# hemis      = ["lh", "rh"]
hemis = ["lh", "rh"]
roi_list   = list(range(1, 9+1))  # ROIs 1 through 9

# --- Load & pivot as before ---
distances_df = pd.read_excel("data/distances/roi_spearman_distances.xlsx")
pivot_dict = {}
for hemi in hemis:
    df_hemi = distances_df[distances_df["hemisphere"] == hemi]
    p = df_hemi.pivot_table(index="roi", columns="subject", values="spearman_rho")
    pivot_dict[hemi] = p.reindex(roi_list)

# --- Global vmin/vmax + norm/cmap choice ---
all_vals = np.hstack([p.values.ravel() for p in pivot_dict.values()])
all_vals = all_vals[~np.isnan(all_vals)]
vmin, vmax = all_vals.min(), all_vals.max()

if vmin < 0 < vmax:
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    cmap = "coolwarm"
else:
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = "viridis"

# --- Plot with reserved space for colorbar ---
fig, (ax_lh, ax_rh) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
# make room on the right for a standalone cbar
fig.subplots_adjust(right=0.82, wspace=0.3)

for ax, hemi in zip((ax_lh, ax_rh), hemis):
    im = ax.imshow(pivot_dict[hemi], norm=norm, cmap=cmap,
                   aspect="auto", interpolation="nearest")
    ax.set_title(f"{hemi.capitalize()} Hemisphere", fontsize=14, fontweight="bold")
    ax.set_xticks(np.arange(len(pivot_dict[hemi].columns)))
    ax.set_xticklabels(pivot_dict[hemi].columns, rotation=45, ha="right")
    ax.set_xlabel("Subject")

# explicitly put ROI labels on the left axis
ax_lh.set_yticks(np.arange(len(pivot_dict[hemis[0]].index)))
ax_lh.set_yticklabels(pivot_dict[hemis[0]].index)
ax_lh.set_ylabel("ROI")

# add a new axes for the colorbar (outside the heatmaps)
cbar_ax = fig.add_axes([0.85, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Spearman ρ", rotation=270, labelpad=15)

fig.suptitle("Spearman ρ (MDS vs. Cortical) per ROI × Subject", fontsize=16, y=0.95)
plt.show()


In [ ]:
agg = (
    distances_df
    .groupby(["hemisphere", "roi"])["spearman_rho"]
    .agg(["mean", "sem"])
    .unstack("hemisphere")
)
means = agg["mean"]                  # DataFrame: index=roi, columns=[left, right]
sems  = agg["sem"]                   # same shape, for yerr

# plotting
rois = means.index
x    = np.arange(len(rois))
w    = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, means["lh"],  width=w, label="Left Hemisphere",  yerr=sems["lh"],  capsize=4)
ax.bar(x + w/2, means["rh"], width=w, label="Right Hemisphere", yerr=sems["rh"], capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(rois, rotation=45, ha="right")
ax.set_ylabel("Mean Spearman ρ")
ax.set_title("Average Spearman ρ per ROI and Hemisphere")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize

# subj_list = ["subj_01"]  # adjust as needed
# hemis      = ["lh", "rh"]
hemis = ["lh", "rh"]
roi_list   = list(range(1, 9+1))  # ROIs 1 through 9

# --- Load & pivot as before ---
distances_df = pd.read_excel("data/distances/roi_spearman_distances.xlsx")
pivot_dict = {}
for hemi in hemis:
    df_hemi = distances_df[distances_df["hemisphere"] == hemi]
    p = df_hemi.pivot_table(index="roi", columns="subject", values="spearman_rho")
    pivot_dict[hemi] = p.reindex(roi_list)

# --- Global vmin/vmax + norm/cmap choice ---
all_vals = np.hstack([p.values.ravel() for p in pivot_dict.values()])
all_vals = all_vals[~np.isnan(all_vals)]
vmin, vmax = all_vals.min(), all_vals.max()

if vmin < 0 < vmax:
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    cmap = "coolwarm"
else:
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = "viridis"

# --- Plot with reserved space for colorbar ---
fig, (ax_lh, ax_rh) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
# make room on the right for a standalone cbar
fig.subplots_adjust(right=0.82, wspace=0.3)

for ax, hemi in zip((ax_lh, ax_rh), hemis):
    im = ax.imshow(pivot_dict[hemi], norm=norm, cmap=cmap,
                   aspect="auto", interpolation="nearest")
    ax.set_title(f"{hemi.capitalize()} Hemisphere", fontsize=14, fontweight="bold")
    ax.set_xticks(np.arange(len(pivot_dict[hemi].columns)))
    ax.set_xticklabels(pivot_dict[hemi].columns, rotation=45, ha="right")
    ax.set_xlabel("Subject")

# explicitly put ROI labels on the left axis
ax_lh.set_yticks(np.arange(len(pivot_dict[hemis[0]].index)))
ax_lh.set_yticklabels(pivot_dict[hemis[0]].index)
ax_lh.set_ylabel("ROI")

# add a new axes for the colorbar (outside the heatmaps)
cbar_ax = fig.add_axes([0.85, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Spearman ρ", rotation=270, labelpad=15)

fig.suptitle("Spearman ρ (MDS vs. Cortical) per ROI × Subject", fontsize=16, y=0.95)
plt.show()


In [ ]:
agg = (
    distances_df
    .groupby(["hemisphere", "roi"])["spearman_rho"]
    .agg(["mean", "sem"])
    .unstack("hemisphere")
)
means = agg["mean"]                  # DataFrame: index=roi, columns=[left, right]
sems  = agg["sem"]                   # same shape, for yerr

# plotting
rois = means.index
x    = np.arange(len(rois))
w    = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, means["lh"],  width=w, label="Left Hemisphere",  yerr=sems["lh"],  capsize=4)
ax.bar(x + w/2, means["rh"], width=w, label="Right Hemisphere", yerr=sems["rh"], capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(rois, rotation=45, ha="right")
ax.set_ylabel("Mean Spearman ρ")
ax.set_title("Average Spearman ρ per ROI and Hemisphere")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import json

with open(f"data/labels/shared/face_detection_result.json", "r") as f:
    data = json.load(f)

names = [x["file_name"] for x in data]

print(len(names))

print(len(list(set(names))))